# System Dependencies
To get started with Unstructured.io, we need a few system-wide dependencies:

## Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr)
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

## libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

## ROCm / open-source model update

Changes made in this pass:
- **Vision-language model**: `Qwen/Qwen2.5-7B-Instruct` (text-only) was replaced with `Qwen/Qwen2.5-VL-7B-Instruct`, loaded directly via `transformers` (`AutoModelForImageTextToText` + `AutoProcessor`) instead of `HuggingFacePipeline`/`ChatHuggingFace`. The old code called `llm.invoke([HumanMessage(content=[...image_url...])])`, which is an OpenAI-style multimodal format that `ChatHuggingFace` doesn't actually support - it would have failed or silently ignored the images. The notebook now has a `vlm_generate(prompt_text, images_base64)` helper used everywhere text+image content needs to go to the model.
- **dtype**: `bfloat16` instead of `float16` - RDNA3 (RX 7900 XT) supports bf16 natively and it's more stable for generation.
- **Attention backend**: `attn_implementation="sdpa"` instead of the default, since flash-attention-2 isn't prebuilt for ROCm.
- **Bug fix**: in `create_ai_enhanced_summary`, the "YOUR TASK" instructions were indented inside the `for table in tables` loop, so they got duplicated per table and were skipped entirely for chunks with images-only or text-only content. Moved outside the loop so it's always appended once.
- Embedding model (`BAAI/bge-small-en-v1.5` via `HuggingFaceEmbeddings`) was left as-is - it's a standard sentence-transformers model and already ROCm/PyTorch-compatible. Swap in `ibm-granite/granite-embedding-small-english-r2` here too if you want it to match your `RAG_app` project.

**VRAM note**: Qwen2.5-VL-7B-Instruct in bf16 is ~16GB, which fits on the 20GB RX 7900 XT but leaves little headroom - close other GPU workloads (games, overlays) while running this notebook.


In [1]:
#%pip install -Uq "unstructured[all-docs]" 

In [1]:
import json
from typing import List
import warnings
import os
import io
import base64
import importlib
import torch
import tqdm as notebook_tqdm
from PIL import Image

# Unstructured for document parsing (existing library)
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components (used for embeddings + the vector store only;
# the multimodal LLM is driven directly through transformers, see below)
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Vision-language model, loaded directly via transformers so we get full
# control over ROCm device placement and dtype (see next cell)
from transformers import AutoProcessor, AutoModelForImageTextToText

from dotenv import load_dotenv

load_dotenv()

warnings.filterwarnings("ignore")


c:\Users\lovep\miniconda3\envs\ragApp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Setup ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Use GPU if available, else fallback to CPU
DB_PATH = os.getenv("VECTOR_DB_PATH", "db/chroma")  # Vector DB storage path, overridable via .env

print(f"Using {'GPU' if DEVICE == 'cuda' else 'CPU'} for compute.")

Using GPU for compute.


In [3]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",  # must match ingestion_pipline.py
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},  # must match ingestion_pipline.py
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5434.84it/s]


In [4]:
# --- Vision-language model (VLM) ---
# The pipeline needs one model that can reason over TEXT + TABLES + IMAGES together
# (chunk summarisation in create_ai_enhanced_summary, and the final answer step).
# Qwen/Qwen2.5-VL-7B-Instruct is fully open-weight, ships a standard `transformers`
# implementation (no custom CUDA kernels), and runs fine on ROCm 7.2.1 + PyTorch
# through the same code path as CUDA - no flash-attn build required.
#
# Notes for the RX 7900 XT (20GB VRAM):
#   - bfloat16 is used instead of float16: RDNA3 supports bf16 natively and it's
#     more numerically stable for generation than fp16.
#   - attn_implementation="sdpa" avoids flash-attention-2, which isn't prebuilt for
#     ROCm and would otherwise need to be compiled from source.
VLM_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
vlm_model = AutoModelForImageTextToText.from_pretrained(
    VLM_MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)
vlm_model.eval()


def vlm_generate(prompt_text: str, images_base64: List[str] = None, max_new_tokens: int = 512) -> str:
    """Run the local VLM on a text prompt plus zero or more base64-encoded images.

    Replaces the old `llm.invoke([HumanMessage(...)])` call - HuggingFacePipeline /
    ChatHuggingFace don't support the OpenAI-style image_url content blocks that were
    used before, so this talks to the model directly through transformers instead.
    """
    images_base64 = images_base64 or []

    content = [{"type": "text", "text": prompt_text}]
    pil_images = []
    for b64 in images_base64:
        try:
            img = Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")
            pil_images.append(img)
            content.append({"type": "image"})
        except Exception as e:
            print(f"     ⚠️ Skipping unreadable image: {e}")

    messages = [{"role": "user", "content": content}]
    chat_prompt = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = vlm_processor(
        text=[chat_prompt],
        images=pil_images if pil_images else None,
        return_tensors="pt",
    ).to(vlm_model.device)

    with torch.no_grad():
        output_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
        )

    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    response = vlm_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()


Loading weights: 100%|██████████| 729/729 [00:18<00:00, 39.02it/s] 


In [5]:
def partition_document(file_path: str) -> List[Document]:
    "Extract elements from PDF using unstructured."

    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="fast", # previously "hi_res" for high quality, but caused errors with ROCM; "fast" is quicker and good enough for most cases
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF (don't ignore images)
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )

    print(f"Extracted {len(elements)} elements from {file_path}.")
    return elements

file_path = "xarchiv.pdf" 
elements = partition_document(file_path)

list_elem = set([str(type(e)) for e in elements])  # Show the types of elements extracted
list_elem

No languages specified, defaulting to English.


Extracted 393 elements from xarchiv.pdf.


{"<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [6]:
elements[36].to_dict()  # Show the content of a specific element

{'type': 'NarrativeText',
 'element_id': '802f18c694bcba60958f2359fa9a9aa5',
 'text': 'Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512.',
 'metadata': {'coordinates': {'points': ((107.641, 502.58614439999997),
    (107.641, 579.0713408),
    (505.65748784059974, 579.0713408),
    (505.65748784059974, 502.58614439999997)),
   'system': 'PixelSpace',
   'layout_width': 612.0,
   'layout_height': 792.0},
  'filename':

In [7]:
images = [element for element in elements if element.category == "Image"]
print(f"Found {len(images)} images in the document.")

# There are images in the document, but due to dependencies in the unstructured library with ROCM, we arn't able to display them in this environment.
if len(images) > 0:
    print(f"First image element: {images[0].to_dict()}")

Found 0 images in the document.


In [8]:
for element in list_elem:
    print(f"Element type: {element}")
    for e in elements:
        if str(type(e)) == element:
            print(f"  - Content:\n {e.to_dict()}")
            break  # Show only the first instance of each type

Element type: <class 'unstructured.documents.elements.NarrativeText'>
  - Content:
 {'type': 'NarrativeText', 'element_id': '364e32591a70248550ce7f70ee3ce068', 'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.', 'metadata': {'coordinates': {'points': ((124.313, 72.5901232), (124.313, 112.44032319999997), (487.8945424, 112.44032319999997), (487.8945424, 72.5901232)), 'system': 'PixelSpace', 'layout_width': 612.0, 'layout_height': 792.0}, 'filename': 'xarchiv.pdf', 'last_modified': '2026-08-06T01:52:22', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/pdf', 'parent_id': '435f4a1f9ccde8c962b310ac0457d565'}}
Element type: <class 'unstructured.documents.elements.Title'>
  - Content:
 {'type': 'Title', 'element_id': '4fca245e5549c5017c7b8ba41ad7d5e5', 'text': 'g u A 2', 'metadata': {'coordinates': {'points': ((16.34, 258.9200000000001), (16.34,

In [9]:
def create_chunks_by_title(elements: List[Document]) -> List[Document]:
    "Chunk the document elements by title using unstructured's chunk_by_title."

    chunks = chunk_by_title(
        elements, 
        max_characters=3000,  # Maximum characters per chunk
        new_after_n_chars=2400,  # Start a new chunk after this many characters
        combine_text_under_n_chars=500,  # Combine text under this many characters into the previous chunk
    )
    print(f"Created {len(chunks)} chunks")
    return chunks

chunks = create_chunks_by_title(elements)
chunks

Created 33 chunks


In [10]:
chunks[2].to_dict()  # Show the content of the third chunk

{'type': 'CompositeElement',
 'element_id': '2d4c53cf-72e9-45f7-9d43-e4871bfabc1d',
 'text': 'Introduction\n\nRecurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15].\n\nRecurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across exam

In [11]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content using the local VLM"""
    
    try:
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

        # NOTE: previously this block was indented inside the `for` loop above, so it
        # got duplicated once per table and skipped entirely for text/image-only chunks.
        # Moved out here so it's always appended exactly once.
        prompt_text += """
        YOUR TASK:
        Generate a comprehensive, searchable description that covers:

        1. Key facts, numbers, and data points from text and tables
        2. Main topics and concepts discussed  
        3. Questions this content could answer
        4. Visual content analysis (charts, diagrams, patterns in images)
        5. Alternative search terms users might use

        Make it detailed and searchable - prioritize findability over brevity.

        SEARCHABLE DESCRIPTION:"""

        # Send prompt + images straight to the local vision-language model
        return vlm_generate(prompt_text, images, max_new_tokens=512)
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)


🧠 Processing chunks with AI Summaries...
   Processing chunk 1/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 6/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 7/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 8/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Proc

In [12]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "3 2 0 2\\n\\ng u A 2\\n\\n] L C . s c [\\n\\n7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a\\n\\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\\n\\nAttention Is All You Need\\n\\nAshish Vaswani\\u2217 Google Brain avaswani@google.com\\n\\nNoam Shazeer\\u2217 Google Brain noam@google.com\\n\\nNiki Parmar\\u2217 Google Research nikip@google.com\\n\\nJakob Uszkoreit\\u2217 Google Research usz@google.com\\n\\nLlion Jones\\u2217 Google Research llion@google.com", "tables_html": [], "images_base64": []}'}, page_content='3 2 0 2\n\ng u A 2\n\n] L C . s c [\n\n7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a\n\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\nAttention Is All You Need\n\nAshish Vaswani∗ Google B

In [13]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 33 chunks to chunks_export.json


In [14]:
def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [15]:
# After your retrieval
query = "What are the two main components of the Transformer architecture? "
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 3 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': 'Figure 1: The Transformer - model architecture.\n\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.\n\n3.1 Encoder and Decoder Stacks\n\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512.\n\nDecoder: The decoder is 

In [16]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")
    
    print("🎉 Pipeline completed successfully!")
    return db

# Run the complete pipeline

In [17]:
db = run_complete_ingestion_pipeline("xarchiv.pdf")

🚀 Starting RAG Ingestion Pipeline


No languages specified, defaulting to English.


Extracted 393 elements from xarchiv.pdf.
Created 33 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 6/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 7/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 8/33
     Types found: ['text']
     Tables: 0,

In [18]:
# Query the vector store
query = "How many attention heads does the Transformer use, and what is the dimension of each head? "

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        all_images = []
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"

                # Collect images from all chunks to pass to the VLM alongside the text
                all_images.extend(original_data.get("images_base64", []))
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Send prompt + images straight to the local vision-language model
        return vlm_generate(prompt_text, all_images, max_new_tokens=512)
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)


Based on the provided documents, the Transformer uses 8 attention heads. Each head has a dimension of \( \frac{d_{\text{model}}}{h} = 64 \), where \( h \) is the number of heads and \( d_{\text{model}} \) is the dimension of the model's input and output vectors. Therefore, the dimension of each head is 64.
